<a href="https://colab.research.google.com/github/udhaya2823/Retrieval-Augmented-Generation-based-AI-Research-paper-Assistant/blob/Projects/RAG_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
import faiss  # Library for fast similarity search
import fitz  # PyMuPDF for extracting text from PDFs
import torch  # Used for deep learning models
import gradio as gr  # For creating a web interface
import numpy as np  # For numerical operations
from transformers import pipeline  # Hugging Face Transformers for text generation
from sentence_transformers import SentenceTransformer  # Model for creating text embeddings
import pickle  # For saving and loading objects

# Load a pre-trained Sentence Transformer model for generating vector embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Load a pre-trained text generation model (Flan-T5) from Hugging Face
qa_model = pipeline("text2text-generation", model="google/flan-t5-large")

# Define the FAISS index for storing and searching text embeddings
# 384 is the size of the vector generated by the embedding model
dimension = 384
faiss_index = faiss.IndexFlatL2(dimension)

# List to store the original text chunks for reference
metadata_store = []

def extract_text_from_pdf(pdf_path):
    """Extracts text from a given PDF file."""
    text = ""
    with fitz.open(pdf_path) as pdf:  # Open the PDF file
        for page in pdf:  # Loop through each page
            text += page.get_text()  # Extract text from the page
    return text  # Return extracted text

def clean_text(text):
    """Cleans the extracted text by removing extra spaces and special characters."""
    import re  # Regular expressions for text cleaning
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
    text = re.sub(r'[^a-zA-Z0-9.,;?!\s]', '', text)  # Remove special characters
    return text.strip()  # Remove leading and trailing spaces

def chunk_text(text, chunk_size=200):
    """Splits text into smaller chunks of a fixed size."""
    words = text.split()  # Split text into words
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]


def store_in_faiss(chunks):
    """Encodes text chunks into vectors and stores them in FAISS index."""
    global faiss_index, metadata_store  # Use global variables
    vectors = embedding_model.encode(chunks)  # Convert text into embeddings
    vectors = np.array(vectors).astype('float32')  # Convert to float32 for FAISS
    faiss_index.add(vectors)  # Add vectors to FAISS index
    metadata_store.extend(chunks)  # Store text chunks in metadata_store
    return len(chunks)  # Return the number of stored chunks

def retrieve_relevant_chunks(query, top_k=5):
    """Finds the most relevant text chunks based on the query."""
    query_embedding = embedding_model.encode([query]).astype('float32')  # Convert query to vector
    distances, indices = faiss_index.search(query_embedding, top_k)  # Search FAISS index
    return [metadata_store[i] for i in indices[0] if i < len(metadata_store)]  # Return relevant chunks

def generate_answer(query, retrieved_chunks):
    """Uses the text generation model to answer the query based on retrieved chunks."""
    context = "\n".join(clean_text(chunk) for chunk in retrieved_chunks)  # Combine retrieved text
    prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"  # Create input prompt
    response = qa_model(prompt, max_length=512, temperature=0.7, top_p=0.9)  # Generate answer
    return response[0]['generated_text']  # Return generated answer

def process_pdf_and_query(pdf, query):
    """Processes the PDF, retrieves relevant text, and generates an answer."""
    text = extract_text_from_pdf(pdf)  # Extract text from PDF
    cleaned_text = clean_text(text)  # Clean the extracted text
    chunks = chunk_text(cleaned_text)  # Split text into smaller chunks
    store_in_faiss(chunks)  # Store chunks in FAISS index
    retrieved_chunks = retrieve_relevant_chunks(query)  # Get relevant chunks
    return generate_answer(query, retrieved_chunks)  # Generate and return answer

def gradio_interface(pdf, query):
    """Defines the Gradio web interface function."""
    return process_pdf_and_query(pdf.name, query)  # Process and return response

# Create a Gradio interface with file and text input, returning text output
iface = gr.Interface(fn=gradio_interface, inputs=["file", "text"], outputs="text",
                     title="DeepDive AI: Research Paper Insights")

# Launch the Gradio web app
iface.launch()

Device set to use cpu


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://58c7e1f7bf0b017b04.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
